# 제로 코스트 추상화: 클래스와 다형성은 컴파일되면서 사라진다

`@cute.kernel` 본문은 **트레이스 시점에 실행되는 파이썬**입니다(*02_control_flow*의 메타프로그램
아이디어). 그래서 여러분이 쓰는 추상화 — 클래스, 객체, 메서드, 심지어 *다형성*까지 — 는 커널이
만들어지는 동안 파이썬 인터프리터가 풀어버리고, 컴파일 결과에는 **아무것도 더 남지 않습니다**.
디바이스 프로그램은 여러분이 방출(emit)한 산술·메모리 연산 그 자체입니다. 예제 두 개를 보고,
생성된 PTX를 직접 읽어서 이 추상화가 공짜라는 걸 확인합니다.

원문: [`NVIDIA/cutlass` examples/python/CuTeDSL/notebooks/1_dsl_features/04_zero_cost_abstraction.ipynb](https://github.com/NVIDIA/cutlass/blob/098de2a652cf8f00fd70b2df54051c7eccbb855a/examples/python/CuTeDSL/notebooks/1_dsl_features/04_zero_cost_abstraction.ipynb) (BSD-3-Clause, commit `098de2a` 기준)

**배울 내용:** 커널 안에서 쓰는 평범한 파이썬 **클래스 / 객체 / 메서드**는 **제로 코스트**입니다
(트레이스 시점에 풀리고, 디바이스에는 존재하지 않습니다); (가상) 메서드를 공유하는 객체 리스트로 만든
**다형성 파이프라인**이 *언롤되고 인라인되어* 직선형 융합 산술로 바뀐다는 것; 그리고 생성된 **PTX**를
읽는 법(`cute.compile[KeepPTX(True)](...).__ptx__`) — 객체도, vtable도, 디스패치도 없다는 걸
확인합니다.

**실행 환경:** 이 노트북은 *커널을 실제로 launch합니다*(섹션 1은 `cute.printf` 커널, 섹션 2는
`torch` 텐서를 더하고 곱하는 커널). 노트북3과 달리 **`CUTE_DSL_ARCH` 오버라이드가 필요 없습니다** —
쓰는 연산이 printf, load/store, float32 덧셈·곱셈뿐이라 sm_90 전용이 아닙니다. 오버라이드를 걸면
오히려 L4용이 아닌 cubin이 만들어져 launch가 실패할 수 있으니 넣지 마세요. `Arch` enum이 `sm_80`부터
시작하므로 T4는 해당 없습니다(README 참고). 이 노트북은 **Colab L4**에서 실행해 검증했습니다.

## 0. 환경 확인

재현성을 위해 먼저 GPU 정보를 남깁니다.

In [ ]:
!nvidia-smi

## 0-1. CuTe DSL 설치

`nvidia-cutlass-dsl==4.8.0`으로 고정합니다(노트북1과 동일 버전).

**함정(노트북1에서 이미 확인함): 같은 커널에서 설치 직후 import하면 실패합니다.** `.pth`
파일이 인터프리터 시작 시점에만 읽히기 때문 — 설치 셀 다음에 반드시 `site.main()` 셀을
넣습니다.

In [ ]:
!pip install -q nvidia-cutlass-dsl==4.8.0

In [ ]:
import site
site.main()  # 방금 설치된 nvidia-cutlass-dsl의 .pth 경로를 sys.path에 반영

In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.cute import KeepPTX   # lets us read the generated PTX back via __ptx__
from typing import Any
import torch

### 함정: `cute.printf` 출력이 셀에 기본적으로 안 뜹니다 (노트북1·2 참고)

노트북1·2에서 확인한 대로, `cute.printf`(디바이스 출력) 결과가 이 Colab 환경에서는 셀 출력에
기본적으로 나타나지 않습니다(원인은 확인 못 함 — 노트북1의 실측 검증 절 참고). 섹션 1은
`Printer.print`가 `cute.printf`를 호출하고, 원문이 "1이 두 번 찍힌다"를 확인 포인트로 삼으니 그대로 두면
그 줄이 빠집니다. 노트북1·2의 `show_device_printf()` 헬퍼를 그대로 가져와 씁니다.

In [ ]:
import os
import sys
import ctypes
import tempfile
import contextlib


@contextlib.contextmanager
def show_device_printf():
    """cute.printf가 fd 1에 직접 쓰는 내용을 캡처해서 with 블록이 끝난 뒤 print합니다."""
    sys.stdout.flush()
    saved_fd1 = os.dup(1)
    tmp = tempfile.NamedTemporaryFile(delete=False)
    tmp_path = tmp.name
    tmp.close()
    tmp_fd = os.open(tmp_path, os.O_WRONLY)
    os.dup2(tmp_fd, 1)
    try:
        yield
        cutlass.cuda.stream_sync(cutlass.cuda.default_stream())
        ctypes.CDLL(None).fflush(None)
    finally:
        sys.stdout.flush()
        os.dup2(saved_fd1, 1)
        os.close(saved_fd1)
        os.close(tmp_fd)
    with open(tmp_path, "r") as f:
        captured = f.read()
    os.unlink(tmp_path)
    if captured:
        print(captured, end="")

## 1. 클래스는 공짜입니다

`Printer`는 평범한 파이썬 클래스입니다. 커널 안에서 객체를 만들고 그 `.print(val)` 메서드를
**두 번** 부릅니다 — 한 번은 클래스를 직접 이름으로 써서, 한 번은 `Constexpr` 인자로 넘겨받은
클래스를 통해서. 둘 다 호스트가 본문을 트레이스하는 동안 돌아가는 순수 파이썬이고, *방출되는* 건
`.print` 안의 `cute.printf(val)` 하나뿐입니다.

In [ ]:
class Printer:
    def __init__(self, printer_method):
        self.printer_method = printer_method

    def print(self, v):
        self.printer_method(v)


@cute.kernel
def printer_kernel(class_type: cutlass.Constexpr[Any], val: cutlass.Int32):
    pm = cute.printf
    Printer(pm).print(val)        # construct a class, call its method
    class_type(pm).print(val)     # same, via a class passed in as Constexpr


@cute.jit
def printer(class_type: cutlass.Constexpr[Any], val: cutlass.Int32):
    printer_kernel(class_type, val).launch(grid=[1, 1, 1], block=[1, 1, 1])

실행한 다음 PTX를 읽어봅니다. 디바이스 프로그램 전체가 **`vprintf` 호출 두 개**(`.print` 하나당 하나)입니다.
`Printer`는 *맹글링된 커널 이름* 안에만 남아 있습니다 — 컴파일 타임 `Constexpr`로 커널의 정체성에
박힌 것이지, 디바이스 위의 객체가 아닙니다.

In [ ]:
cutlass.cuda.initialize_cuda_context()
with show_device_printf():
    printer(Printer, 1)   # -> prints 1 twice (stream_sync는 헬퍼 안에서 호출)

compiled = cute.compile[KeepPTX(True)](printer, Printer, 1)
ptx = compiled.__ptx__
print(ptx)
print("\n>>> vprintf call sites:", sum(1 for l in ptx.splitlines() if 'call' in l and 'vprintf' in l))

# Expected: the device program is just
#   .visible .entry kernel_cutlass_..._class___main__Printer__0( ... )  <- 'Printer' only in the baked name
#   call.uni (retval0), vprintf, (...);   <- .print(val) #1
#   call.uni (retval0), vprintf, (...);   <- .print(val) #2
# >>> vprintf call sites: 2
# 1
# 1

## 2. 다형성도 공짜입니다

이제 진짜입니다: 원소별 연산의 사슬을 *가상* `apply()`를 가진 작은 클래스 계층으로 씁니다 — 기반
클래스 `Op`와, 그걸 오버라이드하는 `AddN` / `MulN`. 커널은 **빌드 타임 리스트**에 든 연산 객체를 돌면서
각각 `op.apply(x)`를 부릅니다. 리스트도 `for` 루프도 메타프로그래밍입니다: 루프는 트레이스 시점에
언롤되고, 가상 `apply()`는 파이썬에서 풀려 인라인됩니다 — 남는 건 직선형 융합 산술이고, 객체도
vtable도 디스패치도 없습니다.

In [ ]:
class Op:
    def apply(self, x):
        raise NotImplementedError      # 'virtual' -- overridden below


class AddN(Op):
    def __init__(self, n):
        self.n = n

    def apply(self, x):
        return x + self.n


class MulN(Op):
    def __init__(self, k):
        self.k = k

    def apply(self, x):
        return x * self.k


@cute.kernel
def fused_kernel(arr: cutlass.Array, ops: cutlass.Constexpr[Any]):
    tx, _, _ = cute.arch.thread_idx()
    x = arr[tx]
    for op in ops:          # build-time list -> loop UNROLLS at trace time
        x = op.apply(x)     # virtual call resolves in Python -> op INLINES
    arr[tx] = x             # -> straight-line fused arithmetic in the IR


@cute.jit
def run(arr: cutlass.Array, ops: cutlass.Constexpr[Any]):
    fused_kernel(arr, ops).launch(grid=[1, 1, 1], block=[256, 1, 1])

일반 파이썬으로 파이프라인(다형적 연산들의 리스트)을 만들고, 실행해서 결과를 확인합니다. 그다음
융합된 커널의 PTX를 읽어보세요: 세 개의 `apply()` 호출이 **`call` 지점 0개**인 산술 명령어 몇 개로
바뀌어 있습니다. 연산 리스트를 바꾸거나 늘리면 커널은 공짜로 다시 특수화(respecialize)됩니다.

In [ ]:
ops = [AddN(5.0), MulN(2.0), AddN(-1.0)]   # (x + 5) * 2 - 1, built from objects
n = 256
data = torch.arange(n, dtype=torch.float32).cuda()
run(cute.runtime.from_dlpack(data), ops)
cutlass.cuda.stream_sync(cutlass.cuda.default_stream())
ref = (torch.arange(n, dtype=torch.float32).cuda() + 5.0) * 2.0 - 1.0
torch.testing.assert_close(data, ref, atol=1e-3, rtol=1e-3)
print("pipeline PASS")

# Read the fused kernel's PTX -- the op pipeline collapsed to straight-line arithmetic.
data2 = torch.arange(n, dtype=torch.float32).cuda()
ptx = cute.compile[KeepPTX(True)](run, cute.runtime.from_dlpack(data2), ops).__ptx__
for l in ptx.splitlines():
    if any(k in l for k in ('ld.global', 'add.f32', 'mul.f32', 'fma.', 'st.global')):
        print("   ", l.strip())
print(">>> call sites in the fused kernel:", ptx.count('call'))

# Expected output: three ops -> two arithmetic instructions, no calls:
#   ld.global.b32   %r2, [%rd3];
#   add.f32         %r3, %r2, 0f40A00000;                  <- + 5.0
#   fma.rn.f32      %r4, %r3, 0f40000000, 0fBF800000;      <- * 2.0 - 1.0  (fused)
#   st.global.b32   [%rd3], %r4;
# >>> call sites in the fused kernel: 0
# pipeline PASS

## 핵심 정리

다형적인 `apply()` 세 번이 **산술 명령어 두 개와 `call` 0개**로 컴파일됐습니다. `Op` 계층, 가상
디스패치, 연산 리스트를 도는 파이썬 `for` — 전부 트레이스 시점에 파이썬 인터프리터 안에서 돌고
디바이스에는 흔적을 남기지 않았습니다. 이게 *제로 코스트 추상화*입니다: 파이썬 객체 모델을 통째로
써서 커널을 작성하되, 런타임에는 그 어느 것도 대가를 치르지 않습니다. (*02_control_flow*의 메타 루프와
같은 메커니즘 — 커널 본문이 메타프로그램이라는 점 — 을 객체에 적용한 것입니다.)

### 실측 검증 (2026-10-01, Colab L4)

이 노트북 전체를 `.ipynb` 단위로(실제 셀 순서대로 하나의 커널에서) Colab **L4**에서 실행해
확인했습니다. 코드 셀 9개가 전부 에러 없이 통과했습니다(`nvidia-cutlass-dsl==4.8.0`). 노트북3과 달리
**`CUTE_DSL_ARCH` 오버라이드 없이** 기본 감지 그대로 돌렸고, 생성된 PTX 헤더도 `.target sm_89`였습니다.

| 섹션 | 확인한 출력 | 원문 기대 출력과 |
|---|---|---|
| 1 클래스 | `vprintf` 호출 지점 **2개**(`>>> vprintf call sites: 2`), `Printer`는 커널 이름 `kernel_cutlass_printer_kernel_class___main__Printer__0`에만 등장, 디바이스 출력 `1`이 두 줄 | 일치 |
| 2 다형성 | `pipeline PASS`, 융합 커널 PTX가 `ld.global.b32` → `add.f32 ... 0f40A00000` → `fma.rn.f32 ... 0f40000000, 0fBF800000` → `st.global.b32`, `>>> call sites in the fused kernel: 0` | 일치(레지스터 번호 `%r2`~`%r4`, `%rd3`까지 동일) |

**실행 셀 수정:** 섹션 1의 실행 셀에서 `printer(Printer, 1)`만 `with show_device_printf():`로 감쌌고,
원래 그 셀에 있던 `cutlass.cuda.stream_sync(...)`는 헬퍼 안으로 옮겼습니다(노트북2와 같은 방식).
나머지 코드 셀 4개는 원문 그대로입니다. 이 노트북에서 헬퍼 *없이* 돌려서 `1` 두 줄이 실제로 빠지는지는
따로 확인하지 않았습니다 — 노트북1·2에서 확인한 현상을 근거로 미리 적용했습니다.

**출력 순서(관찰):** 디바이스 출력 `1`, `1`이 PTX 전체 덤프보다 *위에* 찍혔습니다. 원문 주석은 `1`, `1`을
맨 끝에 적었지만, 셀 코드 순서 그대로입니다: 헬퍼는 `with` 블록이 끝날 때 캡처한 출력을 `print`하고, PTX 출력은 그 다음
줄의 `print(ptx)`가 합니다. 내용은 같고 순서만 다릅니다.

**검증의 한계:** 위 결과는 L4(sm_89)에서 얻은 것입니다. 다른 아키텍처에서의 PTX는 명령어 인코딩이 다를 수
있어 확인하지 않았습니다.


## 직접 해보기

1. **공짜로 다시 특수화하기.** `ops`를 `[MulN(3.0), AddN(7.0)]`으로 바꿔 다시 실행해 보세요 —
   커널은 안 건드리고 빌드 타임 리스트만 바꿨는데, PTX가 거기에 맞게 바뀝니다.
2. **연산 추가하기.** `apply`가 `x * x`를 돌려주는 `class Square(Op)`를 정의해 리스트에 넣고 PTX를
   다시 읽어보세요: `mul.f32`가 하나 늘고, `call`은 여전히 **0개**입니다.
3. **루프가 언롤되는 걸 보기.** `for` 루프 안에 트레이스 시점 `print(type(op).__name__)`를 넣어 보세요 —
   연산 하나당 한 번씩, 트레이싱 중 파이썬에서 찍힙니다(*02_control_flow*의 메타 루프와 똑같이).
4. **다른 산출물.** `compiled.__sass__`는 `__ptx__`가 PTX를 보여주는 것과 같은 방식으로 최종 SASS를
   보여줍니다.